# Houseplant Health Classification with Amazon Rekognition

In [ ]:
# import libraries
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Setup path relative to where the notebook is
data_dir = Path("data")
healthy_dir = data_dir/"healthy"
unhealthy_dir = data_dir/"unhealthy"

# Confirm if folders exists
print("Healthy folder exists:", healthy_dir.exists())
print("Unhealthy folder exists:", unhealthy_dir.exists())

In [ ]:
# Create a lsit  of image files in each folder
healthy_images = [f for f in healthy_dir.iterdir() if f.suffix == ".jpeg"]
unhealthy_images = [f for f in unhealthy_dir.iterdir() if f.suffix == ".jpeg"]

print(f"Healthy images: {len(healthy_images)}")
print(f"Unhealthy images: {len(unhealthy_images)}")
print(f"Total images: {len(healthy_images) + len(unhealthy_images)}")

In [ ]:
# Dsiplay some sample images
from PIL import Image  

def show_sample_images(image_paths, label, n=5):     # show 5 images
    fig, axes = plt.subplots(1, n, figsize=(15, 3))
    fig.suptitle(f"Sample {label} Images", fontsize=14)

    for i, ax in enumerate(axes):
        img = Image.open(image_paths[i])
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"{label} {i+1}")

    plt.tight_layout()
    plt.show()

show_sample_images(healthy_images, "Healthy")
show_sample_images(unhealthy_images, "Unhealthy")

In [ ]:
# Split data for training and testing
from sklearn.model_selection import train_test_split

all_images = healthy_images + unhealthy_images
labels = [0] * len(healthy_images) + [1] * len(unhealthy_images)

# Split inro train and test
train_images, test_images, train_labels, test_labels = train_test_split(
    all_images,
    labels,
    test_size=0.2,
    stratify=labels,  
    random_state=42
)

print(f"Training images: {len(train_images)}")
print(f"Test images: {len(test_images)}")
print(f"Training labels - Healthy: {train_labels.count(0)}, Unhealthy: {train_labels.count(1)}")
print(f"Test labels - Healthy: {test_labels.count(0)}, Unhealthy: {test_labels.count(1)}")

In [ ]:
# Implement data augmentation to deal with class imbalance
import albumentations as A
import cv2
import numpy as np
from tqdm import tqdm
from PIL import Image

# Define transformation for unhealthy images 
unhealthy_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.RandomGamma(p=0.3),
    A.RandomResizedCrop(size=(224, 224), p=0.3)
])

# Define transformationsf for healthy images(more conservative)
healthy_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
])

def augment_images(image_paths, transform, n_augmentations, save_dir):
    for image_path in tqdm(image_paths):
        img = Image.open(image_path).convert("RGB")
        img_array = np.array(img)

        for i in range(n_augmentations):
            augmented = transform(image=img_array) ["image"]
            augmented_img = Image.fromarray(augmented)

            save_path = save_dir / f"{image_path.stem}_aug_{i+1}.jpeg"
            augmented_img.save(save_path)


train_healthy = [f for f in train_images if f.parent.name == "healthy"]
train_unhealthy = [f for f in train_images if f.parent.name == "unhealthy"]

print(f"Training healthy: {len(train_healthy)}")
print(f"Training unhealthy: {len(train_unhealthy)}")


augment_images(train_healthy, healthy_transform, 1, healthy_dir)
augment_images(train_unhealthy, unhealthy_transform, 3, unhealthy_dir)

print(f"Augmentation complete")
print(f"Healthy images after augmentation: {len([f for f in healthy_dir.iterdir()])}")
print(f"Unhealthy images after augmentation: {len([f for f in unhealthy_dir.iterdir()])}")

In [ ]:
# Create S3 Bucket and Upload images
import boto3
from tqdm import tqdm 

# Create S3 client
s3 = boto3.client('s3', region_name='us-east-1')

BUCKET_NAME = 'houseplant-classifier'
s3.create_bucket(Bucket=BUCKET_NAME)
print(f"Bucket {BUCKET_NAME} created succesfully")

print("Uplading training images...")
train_files = list(healthy_dir.iterdir()) + list(unhealthy_dir.iterdir())
train_files = [f for f in train_files if f.suffix == '.jpeg']

for f in tqdm(train_files):
    label_folder = f.parent.name
    s3_key = f"train/{label_folder}/{f.name}"
    s3.upload_file(str(f), BUCKET_NAME, s3_key)

print("Uploading test images...")
for f, label in tqdm(zip(test_images, test_labels), total=len(test_images)):
    label_folder = "healthy" if label == 0 else "unhealthy"
    s3_key = f"test/{label_folder}/{f.name}"
    s3.upload_file(str(f), BUCKET_NAME, s3_key)

print("All images uploaded succesfully")

In [ ]:
# Create Rekognition client
rekognition = boto3.client('rekognition', region_name='us-east-1')
PROJECT_NAME = 'houseplant-classifier'

response = rekognition.create_project(ProjectName=PROJECT_NAME)
project_arn = response['ProjectArn']

print(f"Project created: {project_arn}")

In [ ]:
# Generate manifest files for Amazon Rekognition
import json
from datetime import datetime

def build_manifest_line(source_ref, label_name):
    return json.dumps({
        "source-ref": source_ref,
        label_name: 1,
        f"{label_name}-metadata": {
            "confidence": 1.0,
            "job-name": "labeling-job",
            "class-name": label_name,
            "human-annotated": "yes",
            "creation-date": "2026-01-01T00:00:00",
            "type": "groundtruth/image-classification"
        }
    })

# build train manifest
train_lines = []
paginator = s3.get_paginator('list_objects_v2')
for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix='train/'):
    for obj in page['Contents']:
        if obj['Key'].endswith('.jpeg') or obj['Key'].endswith('.jpg'):
            label = 'healthy' if '/healthy/' in obj['Key'] else 'unhealthy'
            source_ref = f"s3://{BUCKET_NAME}/{obj['Key']}"
            train_lines.append(build_manifest_line(source_ref, label))

# build test manifest
test_lines = []
for path, label in zip(test_images, test_labels):
    label_name = "healthy" if label == 0 else "unhealthy"
    source_ref = f"s3://{BUCKET_NAME}/test/{path.parent.name}/{path.name}"
    test_lines.append(build_manifest_line(source_ref, label_name))


with open("train.manifest", "w") as f:
    f.write("\n".join(train_lines))

with open("test.manifest", "w") as f:
    f.write("\n".join(test_lines))

print(f"Training manifest entries: {len(train_lines)}")
print(f"Test manifest entries: {len(test_lines)}")

In [ ]:
# Upload  manifests to S3
s3.upload_file("train.manifest", BUCKET_NAME, "manifests/train.manifest")
s3.upload_file("test.manifest", BUCKET_NAME, "manifests/test.manifest")

print("Manifests uploaded to S3 successfully")

In [ ]:
# Create Rekognition datasets from the uploaded manifests 
# and link them to the project

import time

# Create training dataset
train_dataset = rekognition.create_dataset(
    DatasetSource={
        'GroundTruthManifest': {
            'S3Object': {
                'Bucket': BUCKET_NAME,
                'Name': 'manifests/train.manifest'
            }
        }
    },
    DatasetType='TRAIN',
    ProjectArn=project_arn
)

# Create test dataset
test_dataset = rekognition.create_dataset(
    DatasetSource={
        'GroundTruthManifest': {
            'S3Object': {
                'Bucket': BUCKET_NAME,
                'Name': 'manifests/test.manifest'
            }
        }
    },
    DatasetType='TEST',
    ProjectArn=project_arn
)

# Wait for both datasets to be ready
print("Waiting for datasets to be ready...")
for dataset_arn, name in [
    (train_dataset['DatasetArn'], 'TRAIN'),
    (test_dataset['DatasetArn'], 'TEST')
]:
    while True:
        resp = rekognition.describe_dataset(DatasetArn=dataset_arn)
        status = resp['DatasetDescription']['Status']
        print(f"{name}: {status}")
        if status == 'CREATE_COMPLETE':
            break
        elif 'FAILED' in status:
            raise Exception(f"{name} dataset creation failed: {status}")
        time.sleep(10)

print("Datasets ready")

In [ ]:
# Define S3 location for training results
OUTPUT_BUCKET =f"s3://{BUCKET_NAME}/output"

# Begin model training
training_response = rekognition.create_project_version(
    ProjectArn=project_arn,
    VersionName="v1",
    OutputConfig={
        'S3Bucket': BUCKET_NAME,
        'S3KeyPrefix': 'output'
    }
)

model_arn = training_response['ProjectVersionArn']
print(f"Training started: {model_arn}")

In [ ]:
# Checktraining status
response = rekognition.describe_project_versions(
    ProjectArn=project_arn,
    VersionNames=["v1"]
)

status = response['ProjectVersionDescriptions'][0]['Status']
print(f"Training status: {status}")

In [ ]:
# Starting the model/running inference
import time

# start the model
print("Starting model...")
rekognition.start_project_version(
    ProjectVersionArn=model_arn,
    MinInferenceUnits=1
)


while True:
    response = rekognition.describe_project_versions(
        ProjectArn=project_arn,
        VersionNames=["v1"]
    )
    status = response['ProjectVersionDescriptions'][0]['Status']
    print(f"Model stats: {status}")

    if status =='RUNNING':
        print("Model is ready for inference")
        break
    elif status == 'FAILED':
        print("Model failed to start")
        break

    time.sleep(30)

In [ ]:
# Run inference on all test images
predictions = []

for image_path in tqdm(test_images):
    with open(image_path, 'rb') as f:
        image_bytes = f.read()
    
    response = rekognition.detect_custom_labels(
        ProjectVersionArn=model_arn,
        Image={'Bytes': image_bytes},
        MinConfidence=50
    )
    
    # Get top prediction
    if response['CustomLabels']:
        top_label = max(response['CustomLabels'], key=lambda x: x['Confidence'])
        predictions.append({
            'image': image_path.name,
            'prediction': top_label['Name'],
            'confidence': top_label['Confidence']
        })
    else:
        predictions.append({
            'image': image_path.name,
            'prediction': 'unknown',
            'confidence': 0
        })

print("Inference complete")
for p in predictions:
    print(f"{p['image']} → {p['prediction']} ({p['confidence']:.1f}%)")

In [ ]:
# Stop model to avoid charges
rekognition.stop_project_version(
    ProjectVersionArn=model_arn
)

print("Model stopped successfully")

In [ ]:
# Evaluation
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

pred_labels = [0 if p['prediction'] == 'healthy' else 1 for p in predictions]

# Print classification report
print(classification_report(test_labels, pred_labels, target_names=['healthy', 'unhealthy']))


# Confusion matrix
cm = confusion_matrix(test_labels, pred_labels)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['healthy', 'unhealthy'],
            yticklabels=['healthy', 'unhealthy'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()

In [ ]:
# Create IAM role for Lambda
import json

iam = boto3.client('iam')

# Trust policy - allows Lambda to use this role
trust_policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {
                "Service": "lambda.amazonaws.com"
            },
            "Action": "sts:AssumeRole"
        }
    ]
}

role_response = iam.create_role(
    RoleName="houseplant-lambda-role",
    AssumeRolePolicyDocument=json.dumps(trust_policy),
    Description="Role for houseplant classifier Lambda function"
)

role_arn = role_response['Role']['Arn']
print(f"Role created: {role_arn}")

# Attach permissions
policies = [
    'arn:aws:iam::aws:policy/AmazonS3FullAccess',
    'arn:aws:iam::aws:policy/AmazonRekognitionFullAccess',
    'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
]

for policy in policies:
    iam.attach_role_policy(
        RoleName='houseplant-lambda-role',
        PolicyArn=policy
    )
    print(f"Attached: {policy.split('/')[-1]}")

print("IAM role ready")

In [ ]:
# Create the Lambda Function
lambda_code = '''
import boto3
import json
import os
import time
from urllib.parse import unquote_plus

rekognition = boto3.client('rekognition')
s3 = boto3.client('s3')

MODEL_ARN = os.environ['MODEL_ARN']
BUCKET_NAME = os.environ['BUCKET_NAME']
PROJECT_ARN = os.environ['PROJECT_ARN']

def handler(event, context):
    bucket = event['Records'][0]['s3']['bucket']['name']
    key = unquote_plus(event['Records'][0]['s3']['object']['key'])
    print(f"Processing image: {key}")
    
    response = rekognition.describe_project_versions(
        ProjectArn=PROJECT_ARN,
        VersionNames=["v1"]
    )
    status = response['ProjectVersionDescriptions'][0]['Status']
    
    if status != 'RUNNING':
        print("Starting model...")
        rekognition.start_project_version(
            ProjectVersionArn=MODEL_ARN,
            MinInferenceUnits=1
        )
        while True:
            response = rekognition.describe_project_versions(
                ProjectArn=PROJECT_ARN,
                VersionNames=["v1"]
            )
            status = response['ProjectVersionDescriptions'][0]['Status']
            print(f"Model status: {status}")
            if status == 'RUNNING':
                break
            elif status == 'FAILED':
                raise Exception("Model failed to start")
            time.sleep(30)
    else:
        print("Model already running")
    
    try:
        s3_object = s3.get_object(Bucket=bucket, Key=key)
        image_bytes = s3_object['Body'].read()
        
        response = rekognition.detect_custom_labels(
            ProjectVersionArn=MODEL_ARN,
            Image={'Bytes': image_bytes},
            MinConfidence=50
        )
        
        if response['CustomLabels']:
            top_label = max(response['CustomLabels'], key=lambda x: x['Confidence'])
            result = {
                'image': key,
                'prediction': top_label['Name'],
                'confidence': top_label['Confidence']
            }
        else:
            result = {
                'image': key,
                'prediction': 'unknown',
                'confidence': 0
            }
        
        result_key = f"results/{key.split('/')[-1].replace('.jpeg', '.json')}"
        s3.put_object(
            Bucket=BUCKET_NAME,
            Key=result_key,
            Body=json.dumps(result)
        )
        
        print(f"Result saved: {result}")
        return result
        
    finally:
        rekognition.stop_project_version(
            ProjectVersionArn=MODEL_ARN
        )
        print("Model stopped")
'''

with open("lambda_function.py", "w") as f:
    f.write(lambda_code)

print("Lambda code updated")

In [ ]:
# Zip & Deploy to AWS
import zipfile

with zipfile.ZipFile('lambda_function.zip', 'w') as z:
    z.write('lambda_function.py')

lambda_client = boto3.client('lambda', region_name='us-east-1')

with open('lambda_function.zip', 'rb') as f:
    zip_bytes = f.read()

lambda_response = lambda_client.create_function(
    FunctionName='houseplant-classifier',
    Runtime='python3.11',
    Role=role_arn,
    Handler='lambda_function.handler',
    Code={'ZipFile': zip_bytes},
    Timeout=600,
    MemorySize=256,
    Environment={
        'Variables': {
            'MODEL_ARN': model_arn,
            'BUCKET_NAME': BUCKET_NAME,
            'PROJECT_ARN': project_arn
        }
    }
)

lambda_arn = lambda_response['FunctionArn']
print(f"Lambda deployed: {lambda_arn}")

In [ ]:
# Give S3 permission to invoke Lambda
try:
    lambda_client.add_permission(
        FunctionName='houseplant-classifier',
        StatementId='S3InvokeLambda',
        Action='lambda:InvokeFunction',
        Principal='s3.amazonaws.com',
        SourceArn=f'arn:aws:s3:::{BUCKET_NAME}'
    )
    print("Permission granted to S3")
except lambda_client.exceptions.ResourceConflictException:
    print("Permission already exists, skipping")


# Set up S3 trigger
# i.e. watch the uplads/ folder and fire the lamda function whenever a .jpeg file is added
s3.put_bucket_notification_configuration(
    Bucket=BUCKET_NAME,
    NotificationConfiguration={
        'LambdaFunctionConfigurations': [
            {
                'LambdaFunctionArn': lambda_arn,
                'Events': ['s3:ObjectCreated:*'],
                'Filter': {
                    'Key': {
                        'FilterRules': [
                            {
                                'Name': 'prefix',
                                'Value': 'uploads/'
                            },
                            {
                                'Name': 'suffix',
                                'Value': '.jpeg'
                            }
                        ]
                    }
                }
            }
        ]
    }
)
print("S3 trigger configured successfully")

In [ ]:
# Testing Uploads...
# sets wait time to 10 minutes for full pipeline to run
# then checks results folder for output

# Upload test image to trigger Lambda
test_image = test_images[0]  
print(f"Uploading: {test_image.name}")

s3.upload_file(
    str(test_image),
    BUCKET_NAME,
    f"uploads/{test_image.name}"
)

print("Image uploaded - waiting 10 minutes for Lambda to process...")
time.sleep(600)

# Check for result
result_key = f"results/{test_image.name.replace('.jpeg', '.json')}"

try:
    result_object = s3.get_object(Bucket=BUCKET_NAME, Key=result_key)
    result = json.loads(result_object['Body'].read())
    print(f"Pipeline result: {result}")
except Exception as e:
    print(f"Result not found: {e}")

In [ ]:
# Verify model is stopped
# imporant for cost control!
response = rekognition.describe_project_versions(
    ProjectArn=project_arn,
    VersionNames=["v1"]
)
status = response['ProjectVersionDescriptions'][0]['Status']
print(f"Model status: {status}")

In [ ]:
# (OPTIONAL)Cleanup 
# uncomment cleanup() and run this to tear down all AWS resources
def cleanup():
    print("Starting cleanup...")
    
    # Stop model if running
    response = rekognition.describe_project_versions(
        ProjectArn=project_arn,
        VersionNames=["v1"]
    )
    status = response['ProjectVersionDescriptions'][0]['Status']
    if status == 'RUNNING':
        rekognition.stop_project_version(ProjectVersionArn=model_arn)
        print("Model stopped")

    # Delete Lambda function
    try:
        lambda_client.delete_function(FunctionName='houseplant-classifier')
        print("Lambda function deleted")
    except lambda_client.exceptions.ResourceNotFoundException:
        print("Lambda function already deleted")

    # Detach and delete IAM role
    try:
        policies = [
            'arn:aws:iam::aws:policy/AmazonS3FullAccess',
            'arn:aws:iam::aws:policy/AmazonRekognitionFullAccess',
            'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole'
        ]
        for policy in policies:
            iam.detach_role_policy(
                RoleName='houseplant-lambda-role',
                PolicyArn=policy
            )
        iam.delete_role(RoleName='houseplant-lambda-role')
        print("IAM role deleted")
    except iam.exceptions.NoSuchEntityException:
        print("IAM role already deleted")

    print("Cleanup complete")

# Uncomment to run cleanup
# cleanup()